# BNPL Default Prediction. Model Serving

Notebook 05. Build, test and deploy the FastAPI prediction endpoint.

Input: models/champion_xgboost.pkl and reports/data_prep_config.json (from Notebooks 03 and 04)

Output: A live API endpoint deployed on Render.com that returns approve or deny decisions.

What this notebook does:

1. Load the champion model and the config file
2. Build the preprocessing function that mirrors Notebook 03 exactly
3. Test preprocessing on a single sample input
4. Build the FastAPI application
5. Test the API locally
6. Write the final serving code to src/bnpl/serving/
7. Instructions for deploying to Render.com

The key principle throughout: the serving pipeline must apply
transformations in exactly the same order and with exactly the same
parameter values as Notebook 03. The data_prep_config.json file
is the single source of truth for all transformation parameters.
Any difference between training and serving is called training-serving
skew and will silently corrupt predictions.


## Section 1. Load Champion Model and Config

In [1]:
import joblib
import json
import pandas as pd
import numpy as np
import os

MODEL_PATH  = '../models/champion_xgboost.pkl'
CONFIG_PATH = '../reports/data_prep_config.json'

model = joblib.load(MODEL_PATH)

with open(CONFIG_PATH, 'r') as f:
    config = json.load(f)

DECISION_THRESHOLD = config['class_imbalance']['scale_pos_weight']

# Override with the business-constrained threshold from Notebook 04
DECISION_THRESHOLD = 0.45

FEATURE_COLS       = config['final_model_columns']
IMPUTE_VALUES      = config['imputation']['values']
OUTLIER_CAPS       = config['outlier_handling']['caps']
TRAIN_CATEGORIES   = config['encoding']['nominal_categories_from_train']
SUB_GRADE_MAP      = config['encoding']['sub_grade']['mapping']

print(f"Model loaded: {MODEL_PATH}")
print(f"Config loaded: {CONFIG_PATH}")
print(f"Decision threshold: {DECISION_THRESHOLD}")
print(f"Feature count expected by model: {len(FEATURE_COLS)}")

Model loaded: ../models/champion_xgboost.pkl
Config loaded: ../reports/data_prep_config.json
Decision threshold: 0.45
Feature count expected by model: 47


## Section 2. Build The Preprocessing Function

This function takes 17 raw input fields and transforms them into the
47 model-ready columns. It must match Notebook 03 exactly.
The config file provides all transformation parameters so nothing
is hardcoded here.


In [5]:
EMP_LENGTH_MAP = {
    '< 1 year': 0, '1 year': 1, '2 years': 2, '3 years': 3, '4 years': 4,
    '5 years': 5, '6 years': 6, '7 years': 7, '8 years': 8, '9 years': 9,
    '10+ years': 10
}

NOMINAL_COLS = ['home_ownership', 'verification_status', 'purpose']

NUMERIC_FEATURES = ['dti', 'fico_range_low', 'revol_util', 'annual_inc',
                     'loan_amnt', 'int_rate', 'delinq_2yrs', 'inq_last_6mths',
                     'open_acc', 'pub_rec', 'revol_bal']

CAPPED_COLS = ['annual_inc', 'revol_bal', 'revol_util', 'dti']

def preprocess_input(raw_input: dict) -> pd.DataFrame:
    d = pd.Series(raw_input).to_frame().T

    # emp_length text to numeric
    d['emp_length_num'] = d['emp_length'].map(EMP_LENGTH_MAP)

    # numeric imputation and was_missing flags
    # using values calculated from training data in Notebook 03
    for col, fill_val in IMPUTE_VALUES.items():
        base_col = col.replace('_num', '') if col == 'emp_length_num' else col
        if base_col in d.columns:
            d[col + '_was_missing'] = d[base_col].isnull().astype(int)
            d[col] = d[base_col].fillna(fill_val)
        elif col in d.columns:
            d[col + '_was_missing'] = d[col].isnull().astype(int)
            d[col] = d[col].fillna(fill_val)
        else:
            d[col + '_was_missing'] = 0
            d[col] = fill_val

    # sub_grade ordinal encoding
    d['sub_grade_encoded'] = d['sub_grade'].map(SUB_GRADE_MAP)
    if d['sub_grade_encoded'].isnull().any():
        d['sub_grade_encoded'] = d['sub_grade_encoded'].fillna(17)

    # term numeric extraction
    d['term_num'] = d['term'].str.extract(r'(\d+)').astype(float)

    # outlier capping using caps from training data
    for col, cap_val in OUTLIER_CAPS.items():
        src = col
        dst = col + '_capped'
        if src in d.columns:
            d[dst] = d[src].clip(upper=cap_val)
        else:
            d[dst] = cap_val

    # one-hot encoding using categories fixed from training data
    for col, cats in TRAIN_CATEGORIES.items():
        for cat in cats:
            d[f"{col}_{cat}"] = (d[col] == cat).astype(int)

    # select exactly the columns the model expects, in the right order
    for col in FEATURE_COLS:
        if col not in d.columns:
            d[col] = 0

    d[FEATURE_COLS] = d[FEATURE_COLS].apply(pd.to_numeric, errors='coerce').fillna(0)

    return d[FEATURE_COLS]

print("Preprocessing function ready.")

Preprocessing function ready.


## Section 3. Test Preprocessing On A Sample Input

In [6]:
sample_input = {
    "dti": 18.5,
    "fico_range_low": 690,
    "revol_util": 45.2,
    "annual_inc": 68000.0,
    "loan_amnt": 10000.0,
    "int_rate": 12.5,
    "sub_grade": "B3",
    "term": "36 months",
    "emp_length": "5 years",
    "home_ownership": "RENT",
    "verification_status": "Verified",
    "purpose": "debt_consolidation",
    "delinq_2yrs": 0.0,
    "inq_last_6mths": 1.0,
    "open_acc": 8.0,
    "pub_rec": 0.0,
    "revol_bal": 12000.0
}

processed = preprocess_input(sample_input)
print(f"Input fields:  17 raw fields")
print(f"Output shape:  {processed.shape}")
print(f"Expected cols: {len(FEATURE_COLS)}")
print()
print("Sample of processed values:")
print(processed.iloc[0][['sub_grade_encoded', 'term_num', 'dti_capped',
                           'annual_inc_capped', 'home_ownership_RENT',
                           'verification_status_Verified']].to_string())

Input fields:  17 raw fields
Output shape:  (1, 47)
Expected cols: 47

Sample of processed values:
sub_grade_encoded                   7.0
term_num                           36.0
dti_capped                         18.5
annual_inc_capped               68000.0
home_ownership_RENT                 1.0
verification_status_Verified        1.0


C:\Users\Prasanth\AppData\Local\Temp\ipykernel_29248\2279448436.py:27: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  d[col] = d[base_col].fillna(fill_val)
C:\Users\Prasanth\AppData\Local\Temp\ipykernel_29248\2279448436.py:27: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  d[col] = d[base_col].fillna(fill_val)
C:\Users\Prasanth\AppData\Local\Temp\ipykernel_29248\2279448436.py:27: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-

In [7]:
probability = model.predict_proba(processed)[0][1]
decision    = "DENY" if probability >= DECISION_THRESHOLD else "APPROVE"

print(f"Default probability: {probability:.4f}")
print(f"Decision threshold:  {DECISION_THRESHOLD}")
print(f"Decision:            {decision}")
print()
print("Preprocessing and prediction pipeline working correctly.")

Default probability: 0.3347
Decision threshold:  0.45
Decision:            APPROVE

Preprocessing and prediction pipeline working correctly.


## Section 4. Build The FastAPI Application

This is the full production API code. It is written here in the notebook
so you can read and understand it, then Section 6 writes it directly
to src/bnpl/serving/api.py as the actual file that gets deployed.


In [8]:
API_CODE = '''
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import Optional
import joblib
import json
import pandas as pd
import numpy as np
import os
from datetime import datetime

app = FastAPI(
    title="BNPL Default Prediction API",
    description="Predicts default probability for BNPL loan applications",
    version="1.0.0"
)

MODEL_PATH  = os.getenv("MODEL_PATH",  "models/champion_xgboost.pkl")
CONFIG_PATH = os.getenv("CONFIG_PATH", "reports/data_prep_config.json")
THRESHOLD   = float(os.getenv("DECISION_THRESHOLD", "0.45"))

model  = joblib.load(MODEL_PATH)
with open(CONFIG_PATH) as f:
    config = json.load(f)

FEATURE_COLS     = config["final_model_columns"]
IMPUTE_VALUES    = config["imputation"]["values"]
OUTLIER_CAPS     = config["outlier_handling"]["caps"]
TRAIN_CATEGORIES = config["encoding"]["nominal_categories_from_train"]
SUB_GRADE_MAP    = config["encoding"]["sub_grade"]["mapping"]

EMP_LENGTH_MAP = {
    "< 1 year": 0, "1 year": 1, "2 years": 2, "3 years": 3, "4 years": 4,
    "5 years": 5, "6 years": 6, "7 years": 7, "8 years": 8, "9 years": 9,
    "10+ years": 10
}


class LoanApplication(BaseModel):
    dti:                 float = Field(..., ge=0,    description="Debt to income ratio")
    fico_range_low:      float = Field(..., ge=300,  le=850, description="Credit score lower bound")
    revol_util:          float = Field(..., ge=0,    le=100, description="Revolving utilization percent")
    annual_inc:          float = Field(..., ge=0,    description="Annual income in dollars")
    loan_amnt:           float = Field(..., ge=0,    description="Loan amount requested")
    int_rate:            float = Field(..., ge=0,    description="Interest rate percent")
    sub_grade:           str   = Field(...,          description="LendingClub sub grade, e.g. B3")
    term:                str   = Field(...,          description="36 months or 60 months")
    emp_length:          str   = Field(...,          description="Employment length, e.g. 5 years")
    home_ownership:      str   = Field(...,          description="RENT, OWN, MORTGAGE or OTHER")
    verification_status: str   = Field(...,          description="Verified, Source Verified or Not Verified")
    purpose:             str   = Field(...,          description="Loan purpose, e.g. debt_consolidation")
    delinq_2yrs:         float = Field(..., ge=0,    description="Delinquencies in last 2 years")
    inq_last_6mths:      float = Field(..., ge=0,    description="Credit inquiries in last 6 months")
    open_acc:            float = Field(..., ge=0,    description="Number of open credit lines")
    pub_rec:             float = Field(..., ge=0,    description="Number of public records")
    revol_bal:           float = Field(..., ge=0,    description="Current revolving balance")


class PredictionResponse(BaseModel):
    decision:            str
    default_probability: float
    threshold_used:      float
    model_version:       str
    timestamp:           str


def preprocess(raw: dict) -> pd.DataFrame:
    d = pd.Series(raw).to_frame().T

    d["emp_length_num"] = d["emp_length"].map(EMP_LENGTH_MAP)

    for col, fill_val in IMPUTE_VALUES.items():
        base = col.replace("_num", "") if col == "emp_length_num" else col
        src  = base if base in d.columns else col
        d[col + "_was_missing"] = d[src].isnull().astype(int) if src in d.columns else 0
        d[col] = d[src].fillna(fill_val) if src in d.columns else fill_val

    d["sub_grade_encoded"] = d["sub_grade"].map(SUB_GRADE_MAP).fillna(17)
    d["term_num"]          = d["term"].str.extract(r"(\d+)").astype(float)

    for col, cap_val in OUTLIER_CAPS.items():
        d[col + "_capped"] = d[col].clip(upper=cap_val) if col in d.columns else cap_val

    for col, cats in TRAIN_CATEGORIES.items():
        for cat in cats:
            d[f"{col}_{cat}"] = (d[col] == cat).astype(int)

    for col in FEATURE_COLS:
        if col not in d.columns:
            d[col] = 0

    return d[FEATURE_COLS]


@app.get("/health")
def health():
    return {
        "status": "healthy",
        "model_version": "champion_xgboost_v1",
        "threshold": THRESHOLD,
        "timestamp": datetime.utcnow().isoformat()
    }


@app.post("/predict", response_model=PredictionResponse)
def predict(application: LoanApplication):
    try:
        raw         = application.model_dump()
        features    = preprocess(raw)
        probability = float(model.predict_proba(features)[0][1])
        decision    = "DENY" if probability >= THRESHOLD else "APPROVE"

        return PredictionResponse(
            decision            = decision,
            default_probability = round(probability, 4),
            threshold_used      = THRESHOLD,
            model_version       = "champion_xgboost_v1",
            timestamp           = datetime.utcnow().isoformat()
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


@app.get("/")
def root():
    return {
        "message": "BNPL Default Prediction API",
        "docs":    "/docs",
        "health":  "/health",
        "predict": "/predict"
    }
'''

print("API code ready. Section 6 will write this to src/bnpl/serving/api.py")

API code ready. Section 6 will write this to src/bnpl/serving/api.py


<>:79: SyntaxWarning: invalid escape sequence '\d'
<>:79: SyntaxWarning: invalid escape sequence '\d'
C:\Users\Prasanth\AppData\Local\Temp\ipykernel_29248\3266304287.py:79: SyntaxWarning: invalid escape sequence '\d'
  d["term_num"]          = d["term"].str.extract(r"(\d+)").astype(float)


## Section 5. Test The API Logic Without Running A Server

In [9]:
test_cases = [
    {
        "name": "Low risk applicant",
        "input": {
            "dti": 8.0, "fico_range_low": 750, "revol_util": 15.0,
            "annual_inc": 120000.0, "loan_amnt": 5000.0, "int_rate": 7.5,
            "sub_grade": "A2", "term": "36 months", "emp_length": "10+ years",
            "home_ownership": "MORTGAGE", "verification_status": "Verified",
            "purpose": "debt_consolidation", "delinq_2yrs": 0.0,
            "inq_last_6mths": 0.0, "open_acc": 10.0, "pub_rec": 0.0,
            "revol_bal": 5000.0
        }
    },
    {
        "name": "Medium risk applicant",
        "input": {
            "dti": 22.0, "fico_range_low": 660, "revol_util": 60.0,
            "annual_inc": 45000.0, "loan_amnt": 10000.0, "int_rate": 16.5,
            "sub_grade": "D2", "term": "60 months", "emp_length": "2 years",
            "home_ownership": "RENT", "verification_status": "Not Verified",
            "purpose": "credit_card", "delinq_2yrs": 1.0,
            "inq_last_6mths": 3.0, "open_acc": 5.0, "pub_rec": 0.0,
            "revol_bal": 18000.0
        }
    },
    {
        "name": "High risk applicant",
        "input": {
            "dti": 35.0, "fico_range_low": 580, "revol_util": 90.0,
            "annual_inc": 28000.0, "loan_amnt": 15000.0, "int_rate": 25.0,
            "sub_grade": "F3", "term": "60 months", "emp_length": "< 1 year",
            "home_ownership": "RENT", "verification_status": "Not Verified",
            "purpose": "small_business", "delinq_2yrs": 3.0,
            "inq_last_6mths": 5.0, "open_acc": 3.0, "pub_rec": 1.0,
            "revol_bal": 22000.0
        }
    }
]

print("Test results:")
print()
for case in test_cases:
    processed   = preprocess_input(case["input"])
    probability = model.predict_proba(processed)[0][1]
    decision    = "DENY" if probability >= DECISION_THRESHOLD else "APPROVE"
    print(f"  {case['name']}")
    print(f"    Default probability: {probability:.4f}")
    print(f"    Decision:            {decision}")
    print()

print("Sanity check: low risk should APPROVE, high risk should DENY.")

Test results:

  Low risk applicant
    Default probability: 0.0893
    Decision:            APPROVE

  Medium risk applicant
    Default probability: 0.7762
    Decision:            DENY

  High risk applicant
    Default probability: 0.8491
    Decision:            DENY

Sanity check: low risk should APPROVE, high risk should DENY.


C:\Users\Prasanth\AppData\Local\Temp\ipykernel_29248\2279448436.py:27: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  d[col] = d[base_col].fillna(fill_val)
C:\Users\Prasanth\AppData\Local\Temp\ipykernel_29248\2279448436.py:27: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  d[col] = d[base_col].fillna(fill_val)
C:\Users\Prasanth\AppData\Local\Temp\ipykernel_29248\2279448436.py:27: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-

## Section 6. Write The API File To src/

In [10]:
import os

os.makedirs('../src/bnpl/serving', exist_ok=True)

init_path = '../src/bnpl/serving/__init__.py'
if not os.path.exists(init_path):
    with open(init_path, 'w') as f:
        f.write('')

api_path = '../src/bnpl/serving/api.py'
with open(api_path, 'w') as f:
    f.write(API_CODE.strip())

print(f"Written: {api_path}")
print()
print("Verify the file exists and looks correct before deploying.")

Written: ../src/bnpl/serving/api.py

Verify the file exists and looks correct before deploying.


## Section 7. Run The API Locally

Run this in your terminal from the project root. Do not run it as a notebook cell.

```
uvicorn src.bnpl.serving.api:app --reload --port 8000
```

Then test it with a curl command in a second terminal:

```
curl -X POST http://localhost:8000/predict ^
  -H "Content-Type: application/json" ^
  -d "{"dti": 18.5, "fico_range_low": 690, "revol_util": 45.2, "annual_inc": 68000, "loan_amnt": 10000, "int_rate": 12.5, "sub_grade": "B3", "term": "36 months", "emp_length": "5 years", "home_ownership": "RENT", "verification_status": "Verified", "purpose": "debt_consolidation", "delinq_2yrs": 0, "inq_last_6mths": 1, "open_acc": 8, "pub_rec": 0, "revol_bal": 12000}"
```

Or open http://localhost:8000/docs in your browser for the interactive API documentation.

You should see a response like:

```json
{
  "decision": "APPROVE",
  "default_probability": 0.2341,
  "threshold_used": 0.45,
  "model_version": "champion_xgboost_v1",
  "timestamp": "2024-01-15T10:30:00"
}
```


## Section 8. Deploy To Render.com

Follow these steps exactly to get a live URL.

Step 1. Make sure these files are in your GitHub repo:
```
src/bnpl/serving/api.py
models/champion_xgboost.pkl
reports/data_prep_config.json
requirements.txt
```

Step 2. Create requirements.txt if you do not have one:
```
fastapi
uvicorn
xgboost
scikit-learn
pandas
numpy
joblib
```

Step 3. Go to render.com and sign up with your GitHub account.

Step 4. Click New, then Web Service.

Step 5. Connect your GitHub repository.

Step 6. Fill in the settings:
```
Name:            bnpl-prediction-api
Environment:     Python
Build Command:   pip install -r requirements.txt
Start Command:   uvicorn src.bnpl.serving.api:app --host 0.0.0.0 --port $PORT
```

Step 7. Click Create Web Service.

Step 8. Wait 2 to 3 minutes for the build to complete.

Step 9. Render gives you a URL like:
```
https://bnpl-prediction-api.onrender.com
```

Step 10. Test the live endpoint:
```
https://bnpl-prediction-api.onrender.com/health
https://bnpl-prediction-api.onrender.com/docs
```

The /docs URL gives you a full interactive interface to test predictions
directly in your browser without writing any code.

Note on the free tier: Render's free tier sleeps after 15 minutes of
inactivity. The first request after sleep takes 30 to 60 seconds to
respond while the service wakes up. This is acceptable for a portfolio
project. A production system would use a paid tier to stay always on.


## Summary

What this notebook built:

The preprocessing function that transforms 17 raw input fields into
47 model-ready columns using the exact same transformation parameters
as Notebook 03, loaded from data_prep_config.json.

The FastAPI application with input validation, prediction, and a
structured response including decision, probability, threshold and
model version.

Three test cases confirming the pipeline produces sensible predictions
before deployment.

The api.py file written to src/bnpl/serving/ ready for deployment.

Instructions for running locally and deploying to Render.com.

Next steps:

Run the API locally first and confirm predictions look sensible.
Then deploy to Render.com and get a live URL.
Once you have a live URL, come back here.
The monitoring notebook is next, which feeds data through the
deployed model and watches for drift.
